In [15]:
from datasets import load_dataset, load_from_disk
ds = load_from_disk("../training_math/pipeline_output/annotated_data")

In [16]:
import numpy as np
import pandas as pd
df = ds.to_pandas()
df["prompt_hint_light"] = df["prompt"] + "\n" + df["hint_light"]
final_df = pd.DataFrame([[x["prompt"], x["completion"]] for i, x in df.iterrows()],columns =["prompt", "completion"])


In [17]:
from sklearn.model_selection import train_test_split

In [18]:
final_df_train, final_df_test = train_test_split(final_df, test_size=0.1)
import json
with open("data/dataset.json", "w") as f:
    json.dump(final_df_train.reset_index(drop=True).to_dict(orient="records"), f)
with open("data/dataset_test.json", "w") as f:
    json.dump(final_df_test.reset_index(drop=True).to_dict(orient="records"), f)

final_df_train

,prompt,completion
627,Wei has designed a logo for his new company us...,The shaded area is equal to the area of the sq...
2550,Let \[ f(x) =\n\begin{cases}\n-x^2 & \text{if...,\begin{align*}\n(f(f(f(f(1)))))\n&=f(f(f(f(-1)...
2886,The graph of $y = f(x)$ is shown below.\n\n[as...,The graph of $y = f(x - 1)$ is produced by tak...
2210,Simplify $\frac{10a^3}{55a^2}$ when $a=3$.,The 10 in the numerator and the 55 in the deno...
1553,"Let $a,$ $b,$ $c$ be nonzero real numbers. Fi...",We can expand the determinant as follows:\n\be...
...,...,...
1126,What is the value of $\sqrt{36+64}-\sqrt{25-16}$?,"Calculating under each square root first, $\sq..."
1332,Compute the smallest real number $x$ which sat...,"Taking a step back, we notice that the given e..."
204,Rationalize the denominator: $\frac1{2\sqrt7}$.,Multiply both numerator and denominator by $\s...
1723,Compute\n\[\sum_{j = 0}^\infty \sum_{k = 0}^\i...,"Expanding, we get\n\begin{align*}\n3k + j + (k..."


## Experimenting with Similar Datapoint similarity maximization and linearhidden state alignment

In [1]:
import json
with open("data/dataset.json", "r") as f:
    final_df_train = json.load(f)
with open("data/dataset_test.json", "r") as f:
    final_df_test = json.load(f)


In [2]:
## compute embeddings
import numpy as np
from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer("google/embeddinggemma-300m")
embeddings = np.array([embed_model.encode(x["prompt"]) for x in final_df_train])
# ===== Nearest neighbor search =====
similarity = embeddings @ embeddings.T  # cosine matrix

for i, row in enumerate(final_df_train):
    similarity[i, i] = -1  # ignore self
    nearest_idx = similarity[i].argmax().item()
    row["close_prompt"] = final_df_train[nearest_idx]["prompt"]

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

In [7]:
#del embed_model
import torch
torch.cuda.empty_cache()

In [4]:
final_df_train[2]

{'prompt': 'The graph of $y = f(x)$ is shown below.\n\n[asy]\nunitsize(0.5 cm);\n\nreal func(real x) {\n  real y;\n  if (x >= -3 && x <= 0) {y = -2 - x;}\n  if (x >= 0 && x <= 2) {y = sqrt(4 - (x - 2)^2) - 2;}\n  if (x >= 2 && x <= 3) {y = 2*(x - 2);}\n  return(y);\n}\n\nint i, n;\n\nfor (i = -5; i <= 5; ++i) {\n  draw((i,-5)--(i,5),gray(0.7));\n  draw((-5,i)--(5,i),gray(0.7));\n}\n\ndraw((-5,0)--(5,0),Arrows(6));\ndraw((0,-5)--(0,5),Arrows(6));\n\nlabel("$x$", (5,0), E);\nlabel("$y$", (0,5), N);\n\ndraw(graph(func,-3,3),red);\n\nlabel("$y = f(x)$", (3,-2), UnFill);\n[/asy]\n\nWhich is the graph of $y = f(x - 1)$?\n\n[asy]\nunitsize(0.5 cm);\n\npicture[] graf;\nint i, n;\n\nreal func(real x) {\n  real y;\n  if (x >= -3 && x <= 0) {y = -2 - x;}\n  if (x >= 0 && x <= 2) {y = sqrt(4 - (x - 2)^2) - 2;}\n  if (x >= 2 && x <= 3) {y = 2*(x - 2);}\n  return(y);\n}\n\nfor (n = 1; n <= 5; ++n) {\n  graf[n] = new picture;\n  for (i = -5; i <= 5; ++i) {\n    draw(graf[n],(i,-5)--(i,5),gray(0.7));\

In [6]:
with open("data/dataset_close_train.json", "w") as f:
    json.dump(final_df_train, f)
with open("data/dataset_close_test.json", "w") as f:
    json.dump(final_df_test, f)

In [3]:
import json
import math
import argparse
from dataclasses import dataclass
from typing import Dict, List
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup
)

from peft import LoraConfig, get_peft_model

In [ ]:



# =========================
# Dataset
# =========================

class PromptDataset(Dataset):
    def __init__(self, path, tokenizer, max_length=512):
        with open(path, "r") as f:
            self.data = json.load(f)

        self.tokenizer = tokenizer
        self.max_length = max_length

    def encode(self, text):
        return self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

    def __getitem__(self, idx):
        item = self.data[idx]

        prompt = item["prompt"]
        completion = item["completion"]

        full = prompt + completion

        enc_full = self.encode(full)
        enc_prompt = self.encode(prompt)

        input_ids = enc_full["input_ids"].squeeze(0)
        attention_mask = enc_full["attention_mask"].squeeze(0)

        labels = input_ids.clone()

        # Mask prompt tokens
        prompt_len = enc_prompt["attention_mask"].sum().item()
        labels[:prompt_len] = -100

        # Mask padding tokens
        labels[attention_mask == 0] = -100

        # Close prompt handling
        close_input_ids = None
        close_attention_mask = None

        if "close_prompt" in item:
            close_full = item["close_prompt"] + completion
            close_enc = self.encode(close_full)

            close_input_ids = close_enc["input_ids"].squeeze(0)
            close_attention_mask = close_enc["attention_mask"].squeeze(0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "close_input_ids": close_input_ids,
            "close_attention_mask": close_attention_mask,
        }

    def __len__(self):
        return len(self.data)


# =========================
# Custom Losses
# =========================

def internal_collinearity_loss(hidden_states, attention_mask):
    """
    Encourage h_t and h_{t-1} to be collinear
    Only over non-padding tokens
    """
    h1 = hidden_states[:, 1:, :]
    h0 = hidden_states[:, :-1, :]

    mask = attention_mask[:, 1:] * attention_mask[:, :-1]

    cos = F.cosine_similarity(h1, h0, dim=-1)

    masked_cos = cos * mask
    return (1 - masked_cos).sum() / mask.sum().clamp(min=1)


def close_question_alignment_loss(hidden_a, hidden_b):
    ha = hidden_a[:, -1, :]
    hb = hidden_b[:, -1, :]
    cos = F.cosine_similarity(ha, hb, dim=-1)
    return (1 - cos).mean()


# =========================
# Training loop
# =========================

def train(
    model,
    dataloader,
    optimizer,
    scheduler,
    device,
    lambda_internal=0.1,
    lambda_close=0.1,
    use_custom_losses=True
):
    model.train()
    total_loss = 0

    for batch in tqdm(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            output_hidden_states=True
        )

        loss = outputs.loss
        hidden = outputs.hidden_states[-1]

        if use_custom_losses:
            # Internal collinearity (masked)
            internal_loss = internal_collinearity_loss(hidden, attention_mask)
            loss = loss + lambda_internal * internal_loss

            # Close-question alignment
            if batch["close_input_ids"][0] is not None:
                close_ids = batch["close_input_ids"].to(device)
                close_mask = batch["close_attention_mask"].to(device)

                with torch.no_grad():
                    close_out = model(
                        input_ids=close_ids,
                        attention_mask=close_mask,
                        output_hidden_states=True
                    )

                close_hidden = close_out.hidden_states[-1]
                close_loss = close_question_alignment_loss(hidden, close_hidden)
                loss = loss + lambda_close * close_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


# =========================
# Main
# =========================

def main(args):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model_name = "Qwen/Qwen2.5-0.5B-Instruct"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
    )

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj", "k_proj", ],
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model_A = get_peft_model(model, lora_config)
    model_B = get_peft_model(model, lora_config)
    model_A.to(device)
    model_B.to(device)

    dataset = PromptDataset(args.dataset, tokenizer)
    dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=50,
        num_training_steps=len(dataloader) * args.epochs
    )

    print("Training with custom losses...")
    for epoch in range(args.epochs):
        loss = train(
            model_A,
            dataloader,
            optimizer,
            scheduler,
            device,
            use_custom_losses=True
        )
        print(f"[Custom] Epoch {epoch} Loss: {loss:.4f}")
    torch.save(
        model_A.state_dict(),
        f"checkpoints/model_custom.pt"
    )
    print("\nTraining with cross-entropy only...")
    for epoch in range(args.epochs):
        loss = train(
            model_B,
            dataloader,
            optimizer,
            scheduler,
            device,
            use_custom_losses=False
        )
        print(f"[CE] Epoch {epoch} Loss: {loss:.4f}")
    torch.save(
        model_B.state_dict(),
        f"checkpoints/model_baseline.pt"
    )

    
from dataclasses import dataclass
@dataclass
class Args:
    dataset: str = "data/dataset_close_train.json"
    batch_size: int = 4
    lr: float = 1e-4
    epochs: int = 8
args = Args()
main(args)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/srv/conda/envs/notebook/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/srv/conda/envs/notebook/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Training with custom losses...


 11%|█         | 72/675 [00:38<05:19,  1.89it/s]

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", ],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)
model_A = get_peft_model(model, lora_config)
model_A.load_state_dict(torch.load(
        f"checkpoints/model_baseline.pt"
    ))

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

<All keys matched successfully>

In [5]:
model_B = get_peft_model(model, lora_config)
model_B.load_state_dict(torch.load(
        f"checkpoints/model_custom.pt"
    ))

/srv/conda/envs/notebook/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/srv/conda/envs/notebook/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


<All keys matched successfully>

In [6]:
model_A = model_A.to("cuda")
model_B = model_B.to("cuda")

In [7]:
def answer_baseline(x):
    inputs = tokenizer(x['prompt'], return_tensors="pt").to("cuda")
    outputs = model_A.generate(**inputs, temperature=0.2, max_new_tokens=512)
    result = tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
    return result
def answer_custom(x):
    inputs = tokenizer(x['prompt'], return_tensors="pt").to("cuda")
    outputs = model_B.generate(**inputs, temperature=0.2, max_new_tokens=512)
    result = tokenizer.decode(outputs[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)
    return result

In [8]:
answer_baseline({"prompt" : "what is the center of the circle of equation (x-1)^2 + (y+1)^2 = 0"})

'?The center of a circle with equation $a(x-h)^2+b(x-h)(y-k)+c(y-k)^2=d$ is $(h,k)$. So, for our circle, we have $a=1$, $b=1$, and $d=-1$. Therefore, the center of the circle is $\\boxed{(1,-1)}$. [asy]\nunitsize(0.5 cm);\n\ndefaultpen(linewidth(.7pt)+fontsize(8pt));\n\ndotfactor=4;\n\nreal f(real x)\n{\nreturn (x - 1);\n}\n\npath p=(1,-1)--f;\ndraw(p);\n\nlabel("$(-1,1)$",p,NW);\n[/asy]'

In [9]:
answer_custom({"prompt" : "what is the center of the circle of equation (x-1)^2 + (y+1)^2 = 0"})

'?The center of a circle with equation $a(x-h)^2+b(x-h)+c(y-k)^2+d(y-k)=0$ is $(h,k)$. In this case, we have that $a=1$, $b=1$, $c=-1$, and $d=1$. Therefore, the center of our circle is $\\boxed{(1,-1)}$. [asy]\nunitsize(0.5 cm);\n\ndraw(Circle((1,-1),1));\ndot((1,-1));\n[/asy]Note: This problem can be solved without graphing by simply plugging in the values for $h$ and $k$ into the equation. For example, if we plug in $h=1$ and $k=-1$, then we get \\[1^2+( -1 + 1)^2 = 0.\\]'

In [10]:
import pandas as pd
import json
with open("data/dataset_test.json", "r") as f:
    data = json.load(f)
df = pd.DataFrame(data)

In [12]:
df = df.sample(n=100, random_state=42)

In [13]:
from tqdm import tqdm
results_baseline = []
for i, x in tqdm(df.iterrows()):
    result = answer_baseline(x)
    results_baseline.append(result)
df['predicted_baseline'] = results_baseline 

results_custom = []
for i, x in tqdm(df.iterrows()):
    result = answer_custom(x)
    results_custom.append(result)
df['predicted_custom'] = results_custom

100it [28:14, 16.94s/it]
100it [30:01, 18.02s/it]


In [14]:
import os
os.environ["OPENAI_API_KEY"] = "csk-9vn2pvfxkk6ptkwemdhxecvp3tdjtt6yjrr8h49pd26k6jrw"#"csk-8hdkfn23cf2fw28ff65r28ev6dw5tc9weectdpfy5xnrhjdh"
os.environ["OPENAI_BASE_URL"] = "https://api.cerebras.ai/v1"
os.environ["OPENAI_MODEL"] = "gpt-oss-120b"

In [17]:
from openai import OpenAI
import time
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", None) or "dummy", base_url=os.environ.get("OPENAI_BASE_URL", None))
model = os.environ.get("OPENAI_MODEL", None)

from tenacity import retry, wait_fixed
@retry(wait=wait_fixed(60))
def evaluate(query, ground_truth, prediction):
    """
    Compares Ground truth with prediction
    """
    system_prompt = (
        "You are a helpful assistant. Given a problem, its solution and the prediction from a model, "
        "Grade the prediction:\n"
        "- Only focus on final answers from both ground truth and prediction, do not look at the preceding reasoning.\n"
        "- If the final answer are the same between the ground truth and the prediction, answer 1.\n"
        "- If the final answer are different between the ground truth and the prediction, answer 0.\n"
        "Only return the grade number, either 1 or 0."
    )
    
    user_message = f"Problem: {query}\nSolution: {ground_truth}\nPrediction Solution: {prediction}"

    response = client.chat.completions.create(
        model=model, # Or any other model
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0.01,
        extra_body=dict(reasoning_effort="low")
    )
    
    return response.choices[0].message.content

In [ ]:
from tqdm import tqdm
import numpy as np
for i, x in tqdm(df.iterrows()):
        df.loc[i, 'score_baseline'] = evaluate(x["prompt"], x["completion"], x["predicted_baseline"])


61it [01:34,  1.05s/it]

In [ ]:
from tqdm import tqdm
import numpy as np
for i, x in tqdm(df.iterrows()):
        df.loc[i, 'score_custom'] = evaluate(x["prompt"], x["completion"], x["predicted_custom"])


In [ ]:
import pandas as pd
df['score_custom'] = df['score_custom'].fillna(value="nan")
df['score_baseline'] = df['score_baseline'].fillna(value="nan")


In [24]:
print("Score Custom" + str(df[df['score_custom'].isin(["0", "1"])].score_custom.astype(int).mean()))
print("Score Baseline" + str(df[df['score_baseline'].isin(["0", "1"])].score_baseline.astype(int).mean()))

Score Custom0.12
Score Baseline0.1


In [ ]:
df.to_csv("data/inference_results.csv")

In [23]:
1

1